In [2]:
# == Applicability-domain reference export (self-contained) ===================
import json
import numpy as np
import pandas as pd
from collections import defaultdict
from datasets import load_dataset
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")

DATASET_NAME        = "scikit-fingerprints/MoleculeNet_Tox21"
DEPLOYED_MODEL_REPO = "mike-malloy/chemberta-tox21-multitarget-scaffold-20260603_1643"
TASK_COLUMNS = ["NR-AR","NR-AR-LBD","NR-AhR","NR-Aromatase","NR-ER","NR-ER-LBD",
                "NR-PPAR-gamma","SR-ARE","SR-ATAD5","SR-HSE","SR-MMP","SR-p53"]
FP_RADIUS, FP_BITS, K = 2, 2048, 5

# --- reproduce the EXACT scaffold split the model trained on (notebook cell 11) ---
raw = load_dataset(DATASET_NAME)
df_raw = raw["train"].to_pandas()
SMILES_COL = "smiles" if "smiles" in raw["train"][0] else "SMILES"

df = df_raw.copy()
df = df[df[TASK_COLUMNS].notna().any(axis=1)]
processed = pd.DataFrame({"smiles": df[SMILES_COL].values}).reset_index(drop=True)

def _murcko(s):
    m = Chem.MolFromSmiles(s)
    if m is None:
        return f"__unparseable__:{s}"
    sc = MurckoScaffold.MurckoScaffoldSmiles(mol=m, includeChirality=False)
    return sc if sc else f"__empty__:{s}"

scaffold_to_idx = defaultdict(list)
for idx, smi in enumerate(processed["smiles"]):
    scaffold_to_idx[_murcko(smi)].append(idx)
scaffold_sets = sorted(scaffold_to_idx.values(), key=lambda s: (len(s), -s[0]), reverse=True)

n = len(processed); train_cutoff, val_cutoff = 0.70 * n, 0.85 * n
train_idx, val_idx = [], []
for g in scaffold_sets:
    if len(train_idx) + len(g) <= train_cutoff:
        train_idx.extend(g)
    elif len(train_idx) + len(val_idx) + len(g) <= val_cutoff:
        val_idx.extend(g)
train_smiles = processed.iloc[train_idx]["smiles"].tolist()
print(f"Reproduced split: total={n}  train={len(train_smiles)} ({len(train_smiles)/n:.1%})")

# --- fingerprints + data-driven out-of-domain threshold ---
_gen = rdFingerprintGenerator.GetMorganGenerator(radius=FP_RADIUS, fpSize=FP_BITS)
fps, kept = [], []
for s in train_smiles:
    m = Chem.MolFromSmiles(s)
    if m is None:
        continue
    fps.append(_gen.GetFingerprint(m))
    kept.append(s)

loo = []
for i, fp in enumerate(fps):
    sims = DataStructs.BulkTanimotoSimilarity(fp, fps)
    sims[i] = -1.0                                   # exclude self
    loo.append(float(np.mean(sorted(sims, reverse=True)[:K])))
threshold = float(np.percentile(loo, 5))

out = {
    "smiles": kept,
    "threshold": round(threshold, 4),
    "params": {"fp_radius": FP_RADIUS, "fp_bits": FP_BITS, "k": K},
    "provenance": {
        "dataset": DATASET_NAME,
        "split": "Bemis-Murcko scaffold, deterministic 70/15/15 (notebook cell 11)",
        "role": "applicability-domain reference = exact model training molecules",
        "model_repo": DEPLOYED_MODEL_REPO,
        "n_train": len(kept),
        "threshold_rule": "5th percentile of per-molecule leave-one-out mean-top-K Tanimoto",
    },
}
with open("ad_reference_smiles.json", "w") as f:
    json.dump(out, f)

print(f"Wrote ad_reference_smiles.json: {len(kept)} molecules, threshold={threshold:.4f}")
print("Sanity -- leave-one-out mean-top-K similarity distribution:")
for p in (1, 5, 25, 50, 75):
    print(f"  {p:>2}th pct: {np.percentile(loo, p):.3f}")

try:
    from google.colab import files
    files.download("ad_reference_smiles.json")
except Exception:
    print("(not on Colab -- file is in the working directory)")

/Users/michaelmalloy/.pyenv/versions/3.12.4/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Reproduced split: total=7831  train=5481 (70.0%)
Wrote ad_reference_smiles.json: 5481 molecules, threshold=0.2699
Sanity -- leave-one-out mean-top-K similarity distribution:
   1th pct: 0.203
   5th pct: 0.270
  25th pct: 0.392
  50th pct: 0.491
  75th pct: 0.581
(not on Colab -- file is in the working directory)
